# PS-S6E02: Model Blending


This notebook tackles the [**Playground Series – Season 6, Episode 2: Predicting Heart Disease**](https://www.kaggle.com/competitions/playground-series-s6e2), a competition focused on predicting if a patient has heart disease, based on features describing the patient's demographics and selected health metrics.

This notebook blends out-of-fold (OOF) and test-set **probability estimates** from multiple models.  The competition is scored by **ROC AUC**, so the blend operates on continuous scores (probabilities) and evaluates blends using OOF AUC.

## Ensemble Strategy: Hill Climbing with Rank Normalization

We combine model outputs using a simple, effective hill-climbing ensemble (Caruana-style):

- Start from the best single model (highest OOF AUC)
- Iteratively add the model that improves the blended OOF AUC the most
- Convert selection counts into weights

Before blending, we apply **rank normalization** to each model's outputs. Rank transforms are monotonic, so they preserve AUC while putting models on a comparable scale.


## Install Needed Packages

(Usually not needed on Kaggle; keep this section only if you add extra dependencies.)


In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

from ps_s06e02_experiment_setup import ExperimentSetup

warnings.filterwarnings('ignore')

%matplotlib inline


In [2]:
helper = ExperimentSetup()

# Get the seed, and apply it to all the internals like pandas and numpy
seed = helper.set_seeds()

helper.configure_pandas()
helper.suppress_warnings()

# Target column in the training dataset
TARGET = 'Heart Disease'


Random seed set to: 10301
Warnings suppressed.


In [3]:
# Load training data and ground truth
training_df = helper.read_dataset('training')

y_true = (
    training_df[TARGET]
    .map({'Absence': 0, 'Presence': 1})
    .astype(np.int8)
    .values
)

train_ids = training_df['id'].values


TRAINING DATASET

   id  Age  Sex  Chest pain type   BP  Cholesterol  FBS over 120  EKG results  \
0   0   58    1                4  152          239             0            0   
1   1   52    1                1  125          325             0            2   
2   2   56    0                2  160          188             0            2   
3   3   44    0                3  134          229             0            2   
4   4   58    1                4  140          234             0            2   

   Max HR  Exercise angina  ST depression  Slope of ST  \
0     158                1          3.600            2   
1     171                0          0.000            1   
2     151                0          0.000            1   
3     150                0          1.000            2   
4     125                1          3.800            2   

   Number of vessels fluro  Thallium Heart Disease  
0                        2         7      Presence  
1                        0         3    

In [4]:
# Sample submission (used to get correct submission column name and id order)
submission_df = helper.read_dataset('submission')
SUB_COL = [c for c in submission_df.columns if c != 'id'][0]


## Loading OOF and Test Probabilities

We load the out-of-fold (OOF) and test-set **probabilities** generated by individual model notebooks.

Expected file naming:
- `*_oof_probs.csv` contains columns: `id`, one probability column (e.g. `prob_xgb`), and optionally `target`
- `*_test_probs.csv` contains columns: `id` and one probability column

All files should share the same `id` values as the competition datasets.


In [5]:
oof_files = []
test_files = []

# Local convention: predictions stored under predictions/s06e02/
# Kaggle convention: mount dataset input that contains predictions/
if helper.running_in_kaggle():
    pred_dir = '/kaggle/input/ps-s06e02-*/predictions'
else:
    pred_dir = 'predictions'

oof_files = sorted(glob.glob(f'{pred_dir}/*_oof_probs.csv'))
test_files = sorted(glob.glob(f'{pred_dir}/*_test_probs.csv'))

print(f'Found {len(oof_files)} OOF files and {len(test_files)} Test files.')
if len(oof_files) > 0:
    print('OOF files:')
    for f in oof_files:
        print(' -', os.path.basename(f))
if len(test_files) > 0:
    print('Test files:')
    for f in test_files:
        print(' -', os.path.basename(f))


Found 4 OOF files and 4 Test files.
OOF files:
 - catboost_oof_probs.csv
 - lgb_oof_probs.csv
 - nn-tabular-resnet_oof_probs.csv
 - xgb_oof_probs.csv
Test files:
 - catboost_test_probs.csv
 - lgb_test_probs.csv
 - nn-tabular-resnet_test_probs.csv
 - xgb_test_probs.csv


In [6]:
# Helper to load and merge probabilities
def load_probs(file_list, index_col='id'):
    series_list = []
    for file in file_list:
        base = os.path.basename(file)
        model_name = (
            base.replace('_oof_probs.csv', '')
                .replace('_test_probs.csv', '')
        )

        df = pd.read_csv(file)

        # Identify the probability column (exclude id/target)
        ignore = {'id', 'target', TARGET}
        prob_cols = [c for c in df.columns if c not in ignore]
        if len(prob_cols) != 1:
            raise ValueError(f'Expected exactly one prob column in {base}, found: {prob_cols}')

        prob_col = prob_cols[0]
        s = df.set_index(index_col)[prob_col].rename(model_name)
        series_list.append(s)

    return pd.concat(series_list, axis=1)


In [7]:
# Load OOF and Test probability tables
oof_df = load_probs(oof_files) if oof_files else pd.DataFrame()
test_df = load_probs(test_files) if test_files else pd.DataFrame()

# Align y_true to oof_df index via id
y_df = pd.DataFrame({'id': train_ids, 'target': y_true}).set_index('id')

common_ids = y_df.index.intersection(oof_df.index)
if len(common_ids) == 0:
    raise RuntimeError('No overlapping ids between training labels and OOF files.')

oof_df = oof_df.loc[common_ids].sort_index()
y_true_aligned = y_df.loc[oof_df.index, 'target'].values

# Align test_df to submission id order (if available)
if not test_df.empty:
    test_df = test_df.loc[submission_df['id'].values].copy()

print('OOF shape:', oof_df.shape)
print('Test shape:', test_df.shape)


OOF shape: (630000, 4)
Test shape: (270000, 4)


In [8]:
def drop_bad_models(oof_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Drop any model column that has NaNs in OOF or Test.
    Then restrict to the intersection of model columns across OOF/Test.
    """
    bad = set()

    if not oof_df.empty:
        bad |= set(oof_df.columns[oof_df.isna().any(axis=0)])
    if not test_df.empty:
        bad |= set(test_df.columns[test_df.isna().any(axis=0)])

    if bad:
        print("Dropping models with missing values:", sorted(bad))
        oof_df = oof_df.drop(columns=[c for c in bad if c in oof_df.columns], errors="ignore")
        test_df = test_df.drop(columns=[c for c in bad if c in test_df.columns], errors="ignore")

    if test_df.empty:
        common = list(oof_df.columns)
    else:
        common = [c for c in oof_df.columns if c in test_df.columns]

    if not common:
        raise RuntimeError("No common model columns remain after dropping bad models.")

    oof_df = oof_df[common]
    if not test_df.empty:
        test_df = test_df[common]

    return oof_df, test_df

oof_df, test_df = drop_bad_models(oof_df, test_df)

print("Final model set:", list(oof_df.columns))
print("OOF shape:", oof_df.shape, "Test shape:", test_df.shape)


Final model set: ['catboost', 'lgb', 'nn-tabular-resnet', 'xgb']
OOF shape: (630000, 4) Test shape: (270000, 4)


In [9]:
# Rank-normalize model outputs to a common scale
def rank_normalize(df: pd.DataFrame) -> pd.DataFrame:
    # maps to (0,1) via average rank; preserves ordering and therefore AUC
    n = len(df)
    return df.rank(method='average') / (n + 1.0)

oof_ranked = rank_normalize(oof_df)
test_ranked = rank_normalize(test_df) if not test_df.empty else test_df

print('\nIndividual Model ROC AUC (OOF, ranked):')
best_single_model = None
best_single_auc = -np.inf

for model in oof_ranked.columns:
    auc = roc_auc_score(y_true_aligned, oof_ranked[model].values)
    print(f'{model}: {auc:.6f}')
    if auc > best_single_auc:
        best_single_auc = auc
        best_single_model = model

print(f'\nBest model/AUC: {best_single_model}/{best_single_auc:.6f}')



Individual Model ROC AUC (OOF, ranked):
catboost: 0.955556
lgb: 0.955485
nn-tabular-resnet: 0.953233
xgb: 0.955453

Best model/AUC: catboost/0.955556


In [10]:
print("Models loaded:", list(oof_ranked.columns))
print("OOF files:", [os.path.basename(f) for f in oof_files])

Models loaded: ['catboost', 'lgb', 'nn-tabular-resnet', 'xgb']
OOF files: ['catboost_oof_probs.csv', 'lgb_oof_probs.csv', 'nn-tabular-resnet_oof_probs.csv', 'xgb_oof_probs.csv']


In [11]:
corr = oof_ranked.corr(method="spearman")

# Drop near-duplicate models (keep the one with better single-model AUC)
threshold = 0.9995
to_drop = set()
cols = list(corr.columns)

single_auc = {c: roc_auc_score(y_true_aligned, oof_ranked[c].values) for c in cols}

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        a, b = cols[i], cols[j]
        if corr.loc[a, b] >= threshold:
            drop = b if single_auc[a] >= single_auc[b] else a
            to_drop.add(drop)

if to_drop:
    print(f"Dropping {len(to_drop)} near-duplicate models (rho >= {threshold}):", sorted(to_drop))
    oof_ranked = oof_ranked.drop(columns=list(to_drop))
    test_ranked = test_ranked.drop(columns=list(to_drop))

print("Models after de-dup:", list(oof_ranked.columns))
display(oof_ranked.corr(method="spearman"))


Models after de-dup: ['catboost', 'lgb', 'nn-tabular-resnet', 'xgb']


,catboost,lgb,nn-tabular-resnet,xgb
catboost,1.000,0.999,0.993,0.999
lgb,0.999,1.000,0.991,0.999
nn-tabular-resnet,0.993,0.991,1.000,0.992
xgb,0.999,0.999,0.992,1.000


## Hill Climbing Algorithm

We search for a weighted ensemble that maximizes OOF ROC AUC. At each iteration we add the single model that most improves the blended AUC.


In [12]:
# Hill Climbing Algorithm (Caruana-style)
# Iteratively adds the model that maximizes the ensemble ROC AUC
def hill_climbing(oof_prob_df: pd.DataFrame, y: np.ndarray, iterations: int = 200, verbose: bool = False):
    if oof_prob_df.isna().any().any():
        bad = oof_prob_df.isna().sum().sort_values(ascending=False)
        raise ValueError(f'NaNs in oof_prob_df:\n{bad[bad>0]}')
    if np.isnan(y).any():
        raise ValueError('NaNs in y')

    # Best single model
    best_single_model = None
    best_single_auc = -np.inf
    for col in oof_prob_df.columns:
        auc = roc_auc_score(y, oof_prob_df[col].values)
        if np.isfinite(auc) and auc > best_single_auc:
            best_single_auc = auc
            best_single_model = col
    if best_single_model is None:
        raise RuntimeError('No finite single-model AUC found (check probs/y).')

    current_sum = oof_prob_df[best_single_model].values.copy()
    k = 1

    model_counts = {c: 0 for c in oof_prob_df.columns}
    model_counts[best_single_model] = 1
    history = [best_single_auc]

    print(f'\nStarting Hill Climbing for {iterations} iterations...')
    print(f'Starting from: {best_single_model} (AUC={best_single_auc:.6f})')

    for i in range(iterations):
        best_step_auc = -np.inf
        best_step_col = None

        for col in oof_prob_df.columns:
            temp_avg = (current_sum + oof_prob_df[col].values) / (k + 1)
            auc = roc_auc_score(y, temp_avg)

            if np.isfinite(auc) and auc > best_step_auc:
                best_step_auc = auc
                best_step_col = col

        if best_step_col is None:
            raise RuntimeError(f'No finite candidate at iter {i+1}.')

        current_sum += oof_prob_df[best_step_col].values
        k += 1
        model_counts[best_step_col] += 1
        history.append(best_step_auc)

        if verbose and ((i + 1) % 10 == 0):
            print(f'Iter {i+1}: +{best_step_col} -> AUC {best_step_auc:.6f}')

    total = sum(model_counts.values())
    weights = {m: c / total for m, c in model_counts.items() if c > 0}
    return weights, history


In [13]:
# Run Optimization
weights, history = hill_climbing(oof_ranked, y_true_aligned, iterations=200, verbose=True)

print('\nFinal Weights:')
for m, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f'{m}: {w:.4f}')

print(f'\nFinal blended OOF AUC: {history[-1]:.6f}')



Starting Hill Climbing for 200 iterations...
Starting from: catboost (AUC=0.955556)
Iter 10: +xgb -> AUC 0.955573
Iter 20: +catboost -> AUC 0.955573
Iter 30: +catboost -> AUC 0.955573
Iter 40: +catboost -> AUC 0.955573
Iter 50: +lgb -> AUC 0.955573
Iter 60: +catboost -> AUC 0.955573
Iter 70: +lgb -> AUC 0.955573
Iter 80: +catboost -> AUC 0.955573
Iter 90: +catboost -> AUC 0.955573
Iter 100: +lgb -> AUC 0.955573
Iter 110: +catboost -> AUC 0.955573
Iter 120: +lgb -> AUC 0.955573
Iter 130: +lgb -> AUC 0.955573
Iter 140: +catboost -> AUC 0.955573
Iter 150: +lgb -> AUC 0.955573
Iter 160: +catboost -> AUC 0.955573
Iter 170: +lgb -> AUC 0.955573
Iter 180: +lgb -> AUC 0.955573
Iter 190: +lgb -> AUC 0.955573
Iter 200: +catboost -> AUC 0.955573

Final Weights:
catboost: 0.6667
lgb: 0.2935
xgb: 0.0398

Final blended OOF AUC: 0.955573


In [14]:
print("OOF columns:", list(oof_df.columns))
print("TEST columns:", list(test_df.columns))

OOF columns: ['catboost', 'lgb', 'nn-tabular-resnet', 'xgb']
TEST columns: ['catboost', 'lgb', 'nn-tabular-resnet', 'xgb']


## Final Ensemble & Submission

We apply the learned weights to both OOF and test probabilities, then write a submission file using the competition’s sample submission schema.


In [15]:
# Blend OOF probabilities (weighted average)
blend_oof = np.zeros(len(oof_ranked), dtype=np.float64)

for model, w in weights.items():
    blend_oof += oof_ranked[model].to_numpy(dtype=np.float64) * float(w)

blend_auc = roc_auc_score(y_true_aligned, blend_oof)
print(f'Blended OOF ROC AUC: {blend_auc:.6f}')


Blended OOF ROC AUC: 0.955573


In [16]:
# Apply weights to Test probabilities
final_test_probs = np.zeros(len(test_ranked), dtype=np.float64)

for model, weight in weights.items():
    final_test_probs += test_ranked[model].to_numpy(dtype=np.float64) * float(weight)

# Save Submission
out = submission_df.copy()
out[SUB_COL] = final_test_probs

out.to_csv('submission.csv', index=False)
print('Saved: submission.csv')

print('SUBMISSION PREVIEW')
print('==================')
print(out.head(10))


Saved: submission.csv
SUBMISSION PREVIEW
       id  Heart Disease
0  630000          0.786
1  630001          0.075
2  630002          0.902
3  630003          0.035
4  630004          0.448
5  630005          0.876
6  630006          0.035
7  630007          0.588
8  630008          0.926
9  630009          0.116
